<a href="https://colab.research.google.com/github/devwoo41/SilJaEon/blob/master/11/W11_practicalNLP_%EC%9D%B4%EC%9A%B0%EC%A3%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Wed May 13 05:20:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 46.3 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset # Huggingface 데이터셋 패키지 import
data = load_dataset("Sp1786/multiclass-sentiment-analysis-dataset") # 데이터 다운로드

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:85: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train_df.csv: 0.00B [00:00, ?B/s]

val_df.csv: 0.00B [00:00, ?B/s]

test_df.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/31232 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5205 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5206 [00:00<?, ? examples/s]

In [4]:
##### 데이터 구조 훑어보기
print("Data type: ", type(data)) # 데이터 타입 확인
print("Data structure: ", data) # 데이터 구조 확인
print("Data keys: ", data.keys()) # 데이터 키 확인
print(data['train'][0]) # 실제 데이터 확인

Data type:  <class 'datasets.dataset_dict.DatasetDict'>
Data structure:  DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'sentiment'],
        num_rows: 31232
    })
    validation: Dataset({
        features: ['id', 'text', 'label', 'sentiment'],
        num_rows: 5205
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'sentiment'],
        num_rows: 5206
    })
})
Data keys:  dict_keys(['train', 'validation', 'test'])
{'id': 9536, 'text': 'Cooking microwave pizzas, yummy', 'label': 2, 'sentiment': 'positive'}


In [5]:
##### 데이터 정제
def remove_empty_data(row):
    return all(row[field] not in [None, ""] for field in ['id', 'text', 'label', 'sentiment'])

# Use the 'filter' method to remove rows with empty data
train_data = data['train'].filter(remove_empty_data)
dev_data = data['validation'].filter(remove_empty_data)
test_data = data['test'].filter(remove_empty_data)

Filter:   0%|          | 0/31232 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5205 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5206 [00:00<?, ? examples/s]

In [6]:
##### 워드 임베딩으로 벡터화 하는 법 구현
### 단어-인덱스 테이블 정의
import numpy as np
import gensim.downloader
import re
import torch

# Gensim 라이브러리를 사용해서 워드 임베딩 불러오기
print(list(gensim.downloader.info()['models'].keys())) # 사용할 수 있는 모델 확인
embeddings = gensim.downloader.load('glove-wiki-gigaword-50')

# load한 embedding 관련 변수 및 Embedding과 관련한 변수 및 단어-인덱스 테이블 정의
emb_words = embeddings.index_to_key # 임베딩에 포함된 단어 리스트
emb_vectors = embeddings.vectors # 임베딩 벡터 리스트
emb_dim = embeddings.vector_size # 임베딩 벡터 차원
emb_stoi = {key: i for i, key in enumerate(emb_words)} # 단어-인덱스 테이블 정의
max_seq_len = 30 # Maximum sequence length

# pad 인덱스: 0 / unk 인덱스 : 1 tjfwjd
emb_stoi['<pad>'] = 0
emb_stoi['<unk>'] = 1

# 사전 학습 된 워드 임베딩에 등록되어 있는 단어에 인덱스 부여
for i, word in enumerate(emb_words):
    emb_stoi[word] = i + 2 # 인덱스 0과 1은 pad와 unk에 할당

# 임베딩 벡터를 텐서 자료형으로 변환
emb_vectors = torch.cuda.FloatTensor(emb_vectors)

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']
[==================================================] 100.0% 66.0/66.0MB downloaded


/tmp/ipykernel_1426/103897746.py:28: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  emb_vectors = torch.cuda.FloatTensor(emb_vectors)


In [7]:
### 데이터를 인덱스로 변환
# 텍스트를 인덱스로 바꾸는 함수
def text_to_index(input_data, stoi, max_seq_len):
    delimiters = r"\s+|(?<=[\w])(?=[,\.!?\'\"\`])|(?<=[,\.!?\'\"\`])(?=[\w'])|(?<=[,\.!?\'\"\`])(?=')"
    all_index = []

    for sample in input_data:
        index_list = []
        word_list = [token for token in re.split(delimiters, sample["text"].lower()) if token]

        for word in word_list:
            if word in stoi.keys(): # 단어가 lookup table에 검색 가능한 경우
                index_list.append(stoi[word])
            else: # 단어가 lookup table에 검색 불가능한 경우
                index_list.append(stoi['<unk>'])

        # Padding : 샘플의 길이가 고정크기의 최대길이(L)보다 작은 경우, pad 인덱스를 추가
        if max_seq_len > len(index_list):
            index_list = index_list + [stoi['<pad>']] * (max_seq_len-len(index_list))
        else: # 샘플의 길이가 고정 크기의 최대길이 보다 큰 경우
            index_list = index_list[:max_seq_len]
        all_index.append(index_list)

    return all_index

# train, dev, test data들을 index로 변환
train_index = text_to_index(train_data, emb_stoi, max_seq_len)
dev_index = text_to_index(dev_data, emb_stoi, max_seq_len)
test_index = text_to_index(test_data, emb_stoi, max_seq_len)

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn

# MLP 모델 정의
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, max_seq_len):
        super(MLP, self).__init__()
        self.input_size = input_size
        self.max_seq_len = max_seq_len
        # 임베딩 계층
        self.word_embedding = nn.Embedding.from_pretrained(embeddings = emb_vectors)
        # 첫번째 은닉층
        self.layer1 = nn.Linear(self.max_seq_len * self.input_size, hidden_size)
        # 두번째 은닉층
        self.layer2 = nn.Linear(hidden_size, hidden_size // 2)
        # 출력층
        self.layer3 = nn.Linear(hidden_size // 2, output_size)
        # 활성화 함수
        self.activation = nn.GELU()
        # 출력층의 활성화 함수
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        # 입력 x를 모델에 적용
        word_emb = self.word_embedding(torch.cuda.LongTensor(x))
        word_emb_new = word_emb.reshape((-1, self.max_seq_len * self.input_size))
        out1 = self.layer1(word_emb_new)
        out2 = self.activation(out1)
        out3 = self.layer2(out2)
        out4 = self.activation(out3)
        out5 = self.layer3(out4)
        out6 = self.activation(out5)
        final_out = self.softmax(out6)

        return final_out

# 하이퍼 파라미터 셋팅
input_size = emb_dim
hidden_size = 100
output_size = 3
learning_rate = 0.001
batch_size = 128
num_epochs = 10
loss_function = nn.CrossEntropyLoss()

# 모델 정의 (초기화)
model = MLP(input_size, hidden_size, output_size, max_seq_len)
device = torch.device("cuda") # use GPU
model = model.to(device)

# 하이퍼파라미터 셋팅 - optimizer 정의
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [9]:
# dev_vectors와 dev_data['label]을 텐서 자료형으로 변환
train_tensors = train_index
dev_tensors = dev_index
dev_labels = torch.tensor(dev_data['label'], dtype=torch.long, device=device)

# 학습 전 dev 성능 확인
dev_outputs = model(dev_tensors)
dev_preds = torch.argmax(dev_outputs, axis=1)
dev_accuracy = torch.sum(dev_preds == dev_labels).item() / len(dev_labels)
print(f"Epoch 0, Accuracy: {dev_accuracy}")

Epoch 0, Accuracy: 0.3675312199807877


In [12]:
best_accuracy = 0
for epoch in range(num_epochs): # 학습 시작 (총 num_epochs 만큼)
    model.train() # 모델을 학습 모드로 설정
    epoch_loss = 0 # epoch마다의 loss를 기록하기 위한 변수
    # batch size 단위로 학습 진행
    for i in range(0, len(train_tensors), batch_size):
        # batch 단위 데이터 생성
        batch_data = train_tensors[i:i+batch_size]
        batch_labels = torch.tensor(train_data['label'][i:i+batch_size], device=device)
        # 1. forward
        outputs = model(batch_data)
        # 2. loss 계산
        loss = loss_function(outputs, batch_labels)
        # 3. 역전파
        optimizer.zero_grad() # 역전파 전에 기울기 초기화
        loss.backward() # 역전파 수행
        # 4. 가중치 업데이트
        optimizer.step() # 가중치 업데이트 수행
        # 현재 batch의 loss 값을 누적 (현재 epoch에 대한 평균 loss 계산용)
        epoch_loss += loss.item()

    # 매 epoch마다 dev 성능 측정
    model.eval()
    with torch.no_grad():
        dev_outputs = model(dev_tensors)
        dev_preds = torch.argmax(dev_outputs, axis=1)
        dev_accuracy = torch.sum(dev_preds == dev_labels).item() / len(dev_labels)

    # save best model on dev data
    if dev_accuracy > best_accuracy:
        best_model = model
        best_accuracy = dev_accuracy
    print(f"Epoch {epoch+1}, Accuracy: {dev_accuracy}, Loss: {epoch_loss /len(train_tensors)}")

Epoch 1, Accuracy: 0.43477425552353505, Loss: 0.0072069711406852624
Epoch 2, Accuracy: 0.4309317963496638, Loss: 0.00707547109742023
Epoch 3, Accuracy: 0.4324687800192123, Loss: 0.00698131292707241
Epoch 4, Accuracy: 0.44111431316042266, Loss: 0.0069337328132547315
Epoch 5, Accuracy: 0.4320845341018252, Loss: 0.006950803560998718
Epoch 6, Accuracy: 0.4209414024975985, Loss: 0.006899464178876188
Epoch 7, Accuracy: 0.4086455331412104, Loss: 0.006808182961292198
Epoch 8, Accuracy: 0.4157540826128722, Loss: 0.006745704468797709
Epoch 9, Accuracy: 0.4107588856868396, Loss: 0.006737466569470822
Epoch 10, Accuracy: 0.4161383285302594, Loss: 0.006694365659209549


In [13]:
##### 테스트 세트로 모델 평가하기
from sklearn.metrics import accuracy_score # Accuracy 측정 함수 import
test_tensors = test_index
pred_results = best_model(test_tensors) # 최종 모델로 test 데이터 예측
pred_labels = torch.argmax(pred_results, axis=1)

accuracy = accuracy_score(test_data['label'], pred_labels.tolist()) # 정확도 측정
print("Accuracy: {:.2f}%".format(accuracy*100)) # 정확도 출력

Accuracy: 40.69%
